In [1]:
import pandas as pd
import numpy as np
import time
from scipy.special import gamma
from openpyxl import load_workbook, Workbook

# Read input data
data = pd.read_excel('data/input42u.xlsx')
a = data['a'].values
b = data['b'].values
c = data['c'].values
Minimum_Capacity = data['pmin'].values
Maximum_Capacity = data['pmax'].values
ramp_up = data['rampup'].values
ramp_down = data['rampdown'].values
Unit = data['Unit'].values

penalty_factor = 1e6

# Read power demand data
power_demand_data = pd.read_excel('data/power_demand42u.xlsx')
power_demand = power_demand_data['load'].values

# Initialize an empty list to store results for each demand
all_outputs = []

# Define function to calculate power produced
def calculate_power(P):
    return np.sum(P)

# Define function to calculate fuel cost with penalty
def calculate_fuel_cost_with_penalty(P, a, b, c, demand):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.divide(np.add(a, np.multiply(b, P) + np.multiply(c, squaredP)), 1000)

    # Penalty for power balance constraint violation
    power_difference = np.sum(P) - demand
    penalty_term = penalty_factor * abs(power_difference)

    total_Fuel_Cost = np.sum(Fuel_Cost) + penalty_term
    return total_Fuel_Cost

# Fibonacci sequence function
def fibonacci(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        a, b = 0, 1
        for _ in range(2, n + 1):
            a, b = b, a + b
        return b

# Fibonacci search
def fibonacci_search(func, a, b, tol=1e-6):
    n = 1
    while (b - a) / 2 > tol:
        n += 1
        x1 = a + (fibonacci(n - 2) / fibonacci(n)) * (b - a)
        x2 = a + (fibonacci(n - 1) / fibonacci(n)) * (b - a)
        if func(x1) < func(x2):
            b = x2
        else:
            a = x1
    return (a + b) / 2

# Golden section search
def golden_section_search(func, a, b, tol=1e-6):
    phi = (1 + np.sqrt(5)) / 2  # Golden ratio
    x1 = b - (b - a) / phi
    x2 = a + (b - a) / phi
    while abs(b - a) > tol:
        if func(x1) < func(x2):
            b = x2
        else:
            a = x1
        x1 = b - (b - a) / phi
        x2 = a + (b - a) / phi
    return (a + b) / 2

def hybrid_search(func, a, b, tol=1e-6):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        if (b - a) < 50:
            return golden_section_search(func, a, b, tol)
        else:
            return fibonacci_search(func, a, b, tol)
    else:
        results = []
        for ai, bi in zip(a, b):
            if (bi - ai) < 50:
                results.append(golden_section_search(func, ai, bi, tol))
            else:
                results.append(fibonacci_search(func, ai, bi, tol))
        return np.mean(results)  # Or return the results as needed

# Koch Snowflake Perturbation Function
def koch_snowflake_perturbation(size, iterations=4, scale=1.0):
    def koch_curve(level, x1, y1, x2, y2):
        if level == 0:
            return [(x1, y1), (x2, y2)]
        else:
            x3 = (2 * x1 + x2) / 3
            y3 = (2 * y1 + y2) / 3
            x4 = (x1 + 2 * x2) / 3
            y4 = (y1 + 2 * y2) / 3
            x5 = (x3 + x4) / 2 + np.sqrt(3) * (y3 - y4) / 6
            y5 = (y3 + y4) / 2 + np.sqrt(3) * (x4 - x3) / 6
            return (koch_curve(level - 1, x1, y1, x3, y3) +
                    koch_curve(level - 1, x3, y3, x5, y5) +
                    koch_curve(level - 1, x5, y5, x4, y4) +
                    koch_curve(level - 1, x4, y4, x2, y2))
    
    perturbations = np.zeros((size, 2))
    for i in range(size):
        x1, y1 = np.random.rand(2)
        x2, y2 = np.random.rand(2)
        curve_points = koch_curve(iterations, x1, y1, x2, y2)
        perturbations[i] = curve_points[np.random.randint(len(curve_points))]
    
    return perturbations * scale

# Clone-based Optimization with Koch Snowflakes
num_iterations = 10
best_cost = float('inf')
best_parameters = None

for iteration in range(num_iterations):
    for demand in power_demand:
        # Koch Snowflake for each parameter
        a_step = koch_snowflake_perturbation(len(a)) * 0.1
        b_step = koch_snowflake_perturbation(len(b)) * 0.1
        c_step = koch_snowflake_perturbation(len(c)) * 0.1

        # Update parameters with Koch Snowflake perturbations
        a_clone = np.add(a, a_step[:, 0])
        b_clone = np.add(b, b_step[:, 1])
        c_clone = np.add(c, a_step[:, 1] + b_step[:, 0])  # Example combination

        # Optimize power production using hybrid search
        def cost_function(P):
            return calculate_fuel_cost_with_penalty(P, a_clone, b_clone, c_clone, demand)

        P = hybrid_search(cost_function, Minimum_Capacity, Maximum_Capacity)

        # Enforce capacity constraints
        for _ in range(10):  # Adjust the number of iterations as needed
            P = np.minimum(np.maximum(P, Minimum_Capacity), Maximum_Capacity)
            
            # Adjust for power balance constraint
            power_difference = np.sum(P) - demand
            if power_difference > 0:
                # Reduce overproduced power
                for j in range(len(P)):
                    if P[j] > Minimum_Capacity[j]:
                        reduction = min(P[j] - Minimum_Capacity[j], power_difference)
                        P[j] -= reduction
                        power_difference -= reduction
            elif power_difference < 0:
                # Increase underproduced power
                for j in range(len(P)):
                    if P[j] < Maximum_Capacity[j]:
                        increase = min(Maximum_Capacity[j] - P[j], abs(power_difference))
                        P[j] += increase
                        power_difference += increase

            # Calculate total fuel cost with penalties
            total_Fuel_Cost = calculate_fuel_cost_with_penalty(P, a_clone, b_clone, c_clone, demand)

            # Update best cost and parameters if a new best is found
            if total_Fuel_Cost < best_cost:
                best_cost = total_Fuel_Cost
                best_parameters = (a_clone, b_clone, c_clone)

# Now best_parameters will contain the optimized values of a, b, c
a, b, c = best_parameters

# Optimization loop
all_outputs = []
schedulling = []

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()
    if load_counter == 1:
        previous_output = None
    else:
        previous_output = all_outputs[-1]['Output']['Power Produced (MW)'].values

    best_position = None
    best_cost = np.inf

    for i in range(len(Minimum_Capacity)):
        P = hybrid_search(lambda x: calculate_fuel_cost_with_penalty(x * np.ones_like(Minimum_Capacity), a, b, c, demand), Minimum_Capacity[i], Maximum_Capacity[i])
        
        # Aggressive enforcement of capacity constraints
        for _ in range(10):  # You can adjust the number of iterations as needed
            P = np.minimum(np.maximum(P, Minimum_Capacity), Maximum_Capacity)
            
            # Adjust for power balance constraint
            power_difference = np.sum(P) - demand
            if power_difference > 0:
                # Reduce overproduced power
                for j in range(len(P)):
                    if P[j] > Minimum_Capacity[j]:
                        reduction = min(P[j] - Minimum_Capacity[j], power_difference)
                        P[j] -= reduction
                        power_difference -= reduction
            elif power_difference < 0:
                # Increase underproduced power
                for j in range(len(P)):
                    if P[j] < Maximum_Capacity[j]:
                        increase = min(Maximum_Capacity[j] - P[j], abs(power_difference))
                        P[j] += increase
                        power_difference += increase
            
            cost = calculate_fuel_cost_with_penalty(P * np.ones_like(Minimum_Capacity), a, b, c, demand)
            if cost < best_cost:
                best_cost = cost
                best_position = P.copy()
            else:
                break  # Exit the loop if cost increases
            
    end_time = time.time()

    # Output Results
    output = pd.DataFrame({'Unit': Unit, 'Power Produced (MW)': best_position})
    print("Load Counter: {}".format(load_counter))
    print(f"Output for Power Demand {demand:.2f} MW")
    print(output)
    print(f"Total Power Produced: {np.sum(best_position):.2f} MW")
    print(f'Total Fuel Cost: Rp {best_cost}')
    print(f'Computation Time: {end_time - start_time} seconds')
    print('------------------------------------------------\n')

    all_outputs.append({'Demand': demand, 
                        'Output': output, 
                        'Total Power Produced': np.sum(best_position),
                        'Total Fuel Cost': best_cost,  # No need to sum again, it's already a single value
                        'Computation Time': end_time - start_time})

    # Scheduling
    output_info = {'Demand': power_demand[load_counter - 1]}
    for unit_index, unit in enumerate(Unit):
        output_info[f'Unit {unit} Power Produced'] = best_position[unit_index]
        output_info[f'Unit {unit} Cost'] = best_cost  # Assign total cost directly
    schedulling.append(output_info)

# Calculate ramp rates and save to an additional sheet
ramp_rates = []
generation_limits = []
for i in range(1, len(power_demand)):  # Start from 1 because there's no previous output for the first demand
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    # Check for ramp rate violations
    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': power_demand[i]}
    generation_limit_info = {'Demand': power_demand[i]}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = ramp_rate[unit_index]
        ramp_rate_info[f'Unit {unit} Violation'] = violations[unit_index]
               
        # Check for generation limit violations
        gen_violation = (current_output[unit_index] < Minimum_Capacity[unit_index]) or (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = current_output[unit_index]
        generation_limit_info[f'Unit {unit} Violation'] = gen_violation
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)
# Save results to Excel
output_file_path = 'FS-GSS-CBO-KS.xlsx'
with pd.ExcelWriter(output_file_path) as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(all_outputs).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)
print("All results saved to 'FS-GSS-CBO-KS.xlsx'")

Load Counter: 1
Output for Power Demand 8761.42 MW
    Unit  Power Produced (MW)
0      1           250.000000
1      2           250.000000
2      3           250.000000
3      4           250.000000
4      5           408.000000
5      6           408.000000
6      7           408.000000
7      8           142.424004
8      9           169.999999
9     10           169.999999
10    11           169.999999
11    12            80.000000
12    13            80.000000
13    14           169.999999
14    15           169.999999
15    16           220.000000
16    17           220.000000
17    18           220.000000
18    19            16.000000
19    20            16.000000
20    21           118.500000
21    22           118.500000
22    23           118.500000
23    24           118.500000
24    25           118.500000
25    26           118.500000
26    27           118.500000
27    28           118.500000
28    29           400.000000
29    30           400.000000
30    31           